1. Fake LLM Template

In [107]:
import random

In [108]:
class NakliLLM:
    
    def __init__(self):
        print("LLM Created")

    def predict(self, prompt):
        respons_list = [
            "BBSR is the capital of Odisha",
            "LaLiga is the best football league",
            "AI is the future of technology"
        ]

        return random.choice(respons_list)

In [109]:
llm = NakliLLM()

LLM Created


In [110]:
llm.predict("What is the capital of Odisha?")

'BBSR is the capital of Odisha'

2. Fake Prompt Template

In [111]:
class NakliPrompt:

    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def format(self, input_dict):
        return self.template.format(**input_dict)

In [112]:
template = NakliPrompt(
    template="Write a {length} poem about {topic}",
    input_variables=["length", "topic"]
)

In [113]:
prompt = template.format({"length":"short", "topic":"odisha"})

In [114]:
llm = NakliLLM()

LLM Created


In [115]:
llm.predict(prompt)

'LaLiga is the best football league'

3. Fake LLM Chain (It connects the Fake LLM & Fake Prompt)

In [116]:
class NakliLLMChain:

    def __init__(self, llm, prompt):
        self.llm = llm
        self.prompt = prompt

    def run(self, input_dict):
        final_prompt=self.prompt.format(input_dict)  # to interact with the LLM, we need format method 
        result = self.llm.predict(final_prompt)   # to interact with the LLM, we need predict method
        return result

In [117]:
template = NakliPrompt(
    template="Write a {length} poem about {topic}",
    input_variables=["length", "topic"]
)

In [118]:
llm = NakliLLM()

LLM Created


In [119]:
chain = NakliLLMChain(llm, template)

In [120]:
chain.run({"length":"short", "topic":"odisha"})

'LaLiga is the best football league'

Prompt needs a format method to create the final prompt.

LLM needs a predict/invoke method to generate the output.

Different components use different methods.

Runnable provides a standardized interface to connect and execute these components.

4. Runnable Chain

In [121]:
from abc import ABC, abstractmethod

In [122]:
# abstact base class for all runnables
class Runnable(ABC):

    @abstractmethod
    def invoke(input_data):
        pass

In [123]:
import random
import warnings

class NakliLLM(Runnable):

    def __init__(self):
        print("LLM Created")

    def invoke(self, prompt):
        response_list = [
            "BBSR is the capital of Odisha",
            "LaLiga is the best football league",
            "AI is the future of technology"
        ]
        return {"response": random.choice(response_list)}

    def predict(self, prompt):
        warnings.warn(
            "predict() is deprecated. Use invoke() instead.",
            DeprecationWarning,
            stacklevel=2     # where to show the warning (default=1)
        )
        return self.invoke(prompt)


In [124]:
llm = NakliLLM()

LLM Created


In [125]:
import warnings

class NakliPrompt(Runnable):

    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def invoke(self, input_dict):
        return self.template.format(**input_dict)

    def format(self, input_dict):
        warnings.warn(
            "format() is deprecated. Use invoke() instead.",
            DeprecationWarning,
            stacklevel=2
        )
        return self.template.format(**input_dict)


In [126]:
class NakliStrOutputParser(Runnable):

    def __init__(self):
        pass

    def invoke(self, input_data):
        return input_data["response"]

Chain together

In [127]:
class RunnableConnector(Runnable):

    def __init__(self, runnable_list):
        self.runnable_list = runnable_list

    def invoke(self, input_data):
        for runnable in self.runnable_list:
            input_data = runnable.invoke(input_data)
        return input_data

In [128]:
template = NakliPrompt(
    template="Write a {length} poem about {topic}",
    input_variables=["length", "topic"]
)

In [129]:
llm = NakliLLM()

LLM Created


In [130]:
parser = NakliStrOutputParser()

In [131]:
chain = RunnableConnector([template, llm, parser])

In [132]:
chain.invoke({"length":"short", "topic":"odisha"})

'AI is the future of technology'

5. Chaining Multiple Runnable Chains

In [133]:
template1 = NakliPrompt(
    template = "Write a joke about {topic}",
    input_variables=["topic"]
)

In [134]:
template2 = NakliPrompt(
    template = "Explain the following joke {response}",
    input_variables=["response"]
)

In [135]:
llm = NakliLLM()

LLM Created


In [136]:
parser = NakliStrOutputParser()

In [137]:
chain1 = RunnableConnector([template1, llm])

In [ ]:
chain2 = RunnableConnector([template2, llm, parser])

In [142]:
final_chain = RunnableConnector([chain1, chain2])

In [143]:
final_chain.invoke({"topic":"Football"})

'BBSR is the capital of Odisha'